# Spark version checker

Check Spark versions between this notebook pyspark and spark cluster
Check python versions between this notebook and spark cluster
Check java versions between this book and spark cluster

https://www.oracle.com/java/technologies/javase/jdk17-0-13-later-archive-downloads.html



In [1]:
spark_master = 'spark-master'
spark_workers = ['spark-worker-1', 'spark-worker-2']
python_package_filter = ['pyspark', 'delta-spark']
tfds_root = '/opt/tfds/data'


In [2]:
import subprocess
import datetime
import os
import json

spark_master = 'spark-master'
spark_workers = ['spark-worker-1', 'spark-worker-2']
import re

def extract_version(text: str, marker: str) -> str:
    # Find where the marker first appears
    marker_index = text.find(marker)
    if marker_index == -1:
        return None  # marker not found

    # Slice the text to search only after the marker
    after_marker = text[marker_index + len(marker):]

    # Search for the first version number after the marker
    match = re.search(r'\d+\.\d+\.\d+', after_marker)
    if match:
        return match.group(0)
    return None


def run_command(command, container_name=None):
    """Executes a command inside a Docker container and returns the output."""
    if container_name and container_name!='local':
        command = f"docker exec {container_name} {command}"
    try:
        result = subprocess.run(
            command,
            shell=True,                  # Allows passing the command as a single string
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,    # Redirect stderr to stdout
            text=True,
            check=True
        )
        return result.stdout.strip()
    except subprocess.CalledProcessError as e:
        return f"Error: {e.output.strip()}"


def get_spark_versions(container_name=None)->dict:
    result = {}
    result['spark_host'] = run_command(container_name=container_name, command="hostname")
    output = run_command(container_name=container_name, command="spark-shell --version")
    result['spark'] = extract_version(text=output, marker='version')
    result['scala'] = extract_version(text=output, marker='Scala')
    lines = output.split('\n')
    for l in lines:
        marker_index = l.find('OpenJDK')
        if marker_index != -1:
            result['OpenJDK'] = l[marker_index:]
    java = extract_version(text=output, marker='OpenJDK')
    if not java:
        output = run_command(container_name=container_name, command="java --version")
        java = extract_version(text=output, marker='java')
    result['java'] = java
    return result



def get_python_versions(container_name=None)->dict:
    result = {}
    result['python_host'] = run_command(container_name=container_name, command="hostname")
    output = run_command(container_name=container_name, command='python3 --version')
    result['python'] = extract_version(text=output, marker = 'Python')

    output = run_command(container_name=container_name, command='pip freeze --no-cache-dir')
    packages = {}
    for package in output.split('\n'):
        if not '==' in package:
            continue
        parts = package.split('==')
        packages[parts[0]] = parts[1]
    result['python_packages'] = packages
    return result


def get_all_versions()->list:
    version_dicts = []
    for container in ['local', spark_master] + spark_workers:

        version_dict = get_spark_versions(container_name=container)
        for key, value in get_python_versions(container_name=container).items():
            version_dict[key] = value
        version_dicts.append(version_dict)
    return version_dicts

def compare_versions(dict_list:list):
    keys = list(dict_list[0].keys())
    diff_dict = {}
    same_dict = {}
    for key in keys:
        if key in ('python_packages', 'spark_host', 'python_host'):
            continue
        host_value = {}
        for d in dict_list:
            host = d['spark_host']
            host_value[host] = d[key]
        if len(set(host_value.values())) != 1:
            diff_dict[key] = host_value
        else:
            same_dict[key] = dict_list[0][key]

    return diff_dict, same_dict

def compare_packages(dict_list:list, python_packages):
    keys = list(dict_list[0].keys())
    diff_dict = {}
    same_dict = {}

    for key in python_packages:

        host_value = {}
        for d in dict_list:
            host = d['spark_host']
            pkg_dict = d['python_packages']
            for package, version in d['python_packages'].items():
                if package==key:
                    host_value[host] = version
                    break
            else:
                # make sure the value is different in case none of them have the package
                host_value[host] = f'{host} does not have {key}'

        if len(set(host_value.values())) != 1:
            diff_dict[key] = host_value
        else:
            # we know all of the have it...and in the same version
            same_dict[key] = dict_list[0]['python_packages'][key]

    return diff_dict, same_dict

def check_file_system(data_root):
    # put a file on the local file system and check for it in all places
    folder_name = os.path.join(data_root, 'spark')
    file_name = os.path.join(folder_name, 'testfile_' + str(datetime.datetime.timestamp(datetime.datetime.now(tz=datetime.timezone.utc))))
    result = {}
    try:
        with open(file_name, 'w') as f:
            f.write('this is a testfile from spark_check.ipynb, it can be deleted')
            for host in ['local', spark_master] + spark_workers:
                result[host] = run_command(container_name=host, command=f'ls {file_name}')
            test = set(result.values())
            if len(test)!=1 or list(test)[0] != file_name:
                print("!!! Mounts are misconfigured, the test file was not found in some loctions")
            else:
                print("Mounts are all good, test file found in all locations")
            print(json.dumps(result, indent=4))
    except Exception as x:
        raise
    finally:
        os.remove(file_name)
    return result

In [3]:
versions = get_all_versions()
dv,sv = compare_versions(versions)
dp,sp = compare_packages(versions, python_package_filter)
import json
if len(dv.keys()) == 0:
    print('Versions are all good, same in all locations:')
    print(json.dumps(sv, indent = 4))
else:
    print("!!! Versions mismatch:")
    print('diff:')
    print(json.dumps(dv, indent = 4))
    print('same:')
    print(json.dumps(sv, indent = 4))

if len(dp.keys()) == 0:
    print('Packages are all good, same in all locations:')
    print(json.dumps(sp, indent = 4))
else:
    print("!!! Package mismatch:")
    print('diff:')
    print(json.dumps(dp, indent = 4))
    print('same:')
    print(json.dumps(sp, indent = 4))

out = check_file_system(tfds_root)


Versions are all good, same in all locations:
{
    "spark": "3.5.5",
    "scala": "2.12.18",
    "java": "11.0.26",
    "python": "3.8.10"
}
Packages are all good, same in all locations:
{
    "pyspark": "3.5.5",
    "delta-spark": "3.3.0"
}
Mounts are all good, test file found in all locations
{
    "local": "/opt/tfds/data/spark/testfile_1745429793.879163",
    "spark-master": "/opt/tfds/data/spark/testfile_1745429793.879163",
    "spark-worker-1": "/opt/tfds/data/spark/testfile_1745429793.879163",
    "spark-worker-2": "/opt/tfds/data/spark/testfile_1745429793.879163"
}
